In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from sqlalchemy import create_engine, text, Column, Integer, String, Float, Text
from sqlalchemy.orm import declarative_base, sessionmaker
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output, Image, HTML
import pandas as pd
import json
from pydantic import BaseModel, Field
from typing import List, Optional, Literal
import requests

load_dotenv()

# Check all api key's
try:
    OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
    ELEVENLABS_API_KEY = os.environ["ELEVENLABS_API_KEY"]
    SPOONACULAR_API_KEY = os.environ["SPOONACULAR_API_KEY"]
    print("Loading API keys done.")
except KeyError as e:
    raise EnvironmentError(f"Missing api key: {e}")

# Initialize OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

Loading API keys done.


In [2]:
# Define the database class  
Base = declarative_base()

# # User Profile Table
# class UserProfile(Base):
#     __tablename__ = 'user_profile'

#     id = Column(Integer, primary_key=True)
#     name = Column(String, nullable=False)
#     weight = Column(Float)  # in kg
#     height = Column(Float)  # in cm
#     age = Column(Integer)   # 'Male' / 'Female'
#     gender = Column(String)
#     activity_level = Column(String)         # e.g., 'sedentary', 'active'
#     goal = Column(String)              # e.g., 'weight_loss', 'muscle_gain'
#     dietary_restrictions = Column(String)   # e.g., 'gluten, lactose' (comma separated)

class UserProfile(Base):
    __tablename__ = 'user_profile'
    # Wymuszamy aktualizację struktury tabeli (dla SQLite w notebooku)
    __table_args__ = {'extend_existing': True}

    id = Column(Integer, primary_key=True)
    name = Column(String, nullable=False)
    
    # Dane wejściowe
    weight = Column(Float)
    height = Column(Float)
    age = Column(Integer)
    gender = Column(String)
    activity_level = Column(String)
    goal = Column(String)
    dietary_restrictions = Column(String)
    
    # NOWE KOLUMNY: Wyniki obliczeń (Cache) - żeby nie liczyć co chwilę
    target_calories = Column(Integer, default=2000)
    target_protein = Column(Integer, default=150)
    target_fat = Column(Integer, default=70)
    target_carbs = Column(Integer, default=250)
    
    # --- METODA OOP: PRZELICZANIE CELÓW (Wg Twojego pliku CSV) ---
    def calculate_and_update_targets(self):
        """
        Metoda wywoływana przy zapisie. Przelicza BMR, TDEE i makro.
        """
        # 1. BMR (Mifflin-St Jeor)
        if self.gender == 'Male':
            bmr = (10 * self.weight) + (6.25 * self.height) - (5 * self.age) + 5
        else:
            bmr = (10 * self.weight) + (6.25 * self.height) - (5 * self.age) - 161
            
        # 2. TDEE (Zapotrzebowanie całkowite)
        multipliers = {'sedentary': 1.2, 'moderate': 1.375, 'active': 1.55, 'very_active': 1.725}
        tdee = bmr * multipliers.get(self.activity_level, 1.2)
        
        # 3. Dostosowanie Kalorii do Celu
        if self.goal == 'weight_loss':
            self.target_calories = int(tdee - 400)
        elif self.goal == 'muscle_gain':
            self.target_calories = int(tdee + 300)
        else:
            self.target_calories = int(tdee)

        # 4. Makroskładniki (Wg Twojego pliku CSV)
        # Klucz: (Białko %, Tłuszcze %, Węgle %)
        macro_ratios = {
            'sedentary':    (0.25, 0.35, 0.40), # Mniej węgli, więcej tłuszczu
            'moderate':     (0.30, 0.30, 0.40), # Zbalansowane
            'active':       (0.30, 0.25, 0.45), # Więcej węgli na trening
            'very_active':  (0.25, 0.20, 0.55)  # Paliwo rakietowe (glikogen)
        }
        
        # Pobieramy proporcje dla wybranej aktywności
        p_r, f_r, c_r = macro_ratios.get(self.activity_level, (0.3, 0.3, 0.4))
        
        # Przeliczamy na gramy (Białko/Węgle /4, Tłuszcz /9)
        self.target_protein = int((self.target_calories * p_r) / 4)
        self.target_fat = int((self.target_calories * f_r) / 9)
        self.target_carbs = int((self.target_calories * c_r) / 4)

# Meal History Table (for future tracking)
class MealLog(Base):
    __tablename__ = 'meal_log'

    id = Column(Integer, primary_key=True)
    date = Column(String, default=lambda: datetime.now().isoformat())

    recipe_name = Column(String)
    recipe_url = Column(String)     
    image_url = Column(String)

    calories = Column(Float)
    protein = Column(Float)
    carbs = Column(Float)
    fat = Column(Float)
    
    user_rating = Column(Integer)   # rating 1-10
    user_notes = Column(Text, nullable=True)

# Create nowaste.db file
try:
    UserProfile.__table__.drop(engine)
    print("Tabela UserProfile została zaktualizowana (stare dane profilu usunięte).")
except:
    pass

engine = create_engine('sqlite:///nowaste.db')
Base.metadata.create_all(engine)

# Create session to interact with the database
Session = sessionmaker(bind=engine)
session = Session()

print("Database ready")

Database ready


In [ ]:
# --- Widget Definitions ---
w_name = widgets.Text(description="Name:")
w_weight = widgets.FloatText(description="Weight (kg):", value=70.0)
w_height = widgets.FloatText(description="Height (cm):", value=175.0)
w_age = widgets.IntText(description="Age:", value=30)
w_gender = widgets.Dropdown(options=['Male', 'Female'], description="Gender:", )

# Activity levels mapped to descriptions
w_activity = widgets.Dropdown(
    options=[
        ('Sedentary (Office job, no sports)', 'sedentary'),
        ('Moderate (Exercise 1-3x/week)', 'moderate'),
        ('Active (Exercise 4-5x/week)', 'active'),
        ('Very Active (Athlete)', 'very_active')
    ],
    description="Activity:"
)

# Goals
w_goal = widgets.Dropdown(
    options=[
        ('Weight Loss (Reduction)', 'weight_loss'),
        ('Maintain Weight', 'maintenance'),
        ('Muscle Gain (Bulk)', 'muscle_gain')
    ],
    description="Goal:"
)

w_restrictions = widgets.Textarea(
    description="Intolerances:",
    placeholder="e.g. gluten, peanuts (leave empty if none)",
    
)

btn_save = widgets.Button(description="Save Profile", button_style='success')
output = widgets.Output()

# --- Callback Function to Save Data ---
def save_profile_to_db(b):
    """
    Save user info to database
    """
    with output:
        clear_output()
        
        # Check if a user already exists (assuming single-user app for now)
        existing_user = session.query(UserProfile).first()
        
        if not existing_user:
            existing_user = UserProfile()
            session.add(existing_user)
            print("Creating new profile...")
        else:
            print("Updating existing profile...")
        
        # Assign values from widgets to the database object
        existing_user.name = w_name.value
        existing_user.weight = w_weight.value
        existing_user.height = w_height.value
        existing_user.age = w_age.value
        existing_user.gender = w_gender.value
        existing_user.activity_level = w_activity.value
        existing_user.goal = w_goal.value
        existing_user.dietary_restrictions = w_restrictions.value

        existing_user.calculate_and_update_targets()
        
        # Commit changes to the database
        try:
            session.commit()
            print(f"SUCCESS! Profile for {existing_user.name} saved.")
            print(f"Goal: {existing_user.goal} | Weight: {existing_user.weight}kg")
            print(f"Wyliczone cele: {existing_user.target_calories} kcal")
            print(f"B: {existing_user.target_protein}g | T: {existing_user.target_fat}g | W: {existing_user.target_carbs}g")
        except Exception as e:
            session.rollback()
            print(f"Error saving to database: {e}")

btn_save.on_click(save_profile_to_db)

# --- Display the Form ---
display(widgets.VBox([
    widgets.HTML("<h3>User Profile Setup</h3>"),
    w_name, w_weight, w_height, w_age, w_gender,
    w_activity, w_goal, w_restrictions,
    btn_save, output
]))

In [9]:
def show_table(model_class, table_name):
    stmt = session.query(model_class).statement
    df = pd.read_sql(stmt, session.bind) #type: ignore
    
    if not df.empty:
        display(df)
    else:
        print("(Tabela jest pusta)")
    print("\n")

show_table(UserProfile, "User Profile")

show_table(MealLog, "Meal Log")

,id,name,weight,height,age,gender,activity_level,goal,dietary_restrictions,target_calories,target_protein,target_fat,target_carbs
0,1,Adam,70.0,175.0,30,Male,sedentary,weight_loss,,1578,98,61,157


,id,date,recipe_name,recipe_url,image_url,calories,protein,carbs,fat,user_rating,user_notes
0,1,2026-01-23T20:17:51.916539,Butternut Squash Soup with Fresh Goat Cheese,https://www.foodista.com/recipe/WCCFJNLJ/butte...,https://img.spoonacular.com/recipes/636603-556...,235.79,13.11,18.30,13.21,6,dobra ale nie super
1,2,2026-01-24T12:54:20.473339,Nutella Buttercream Cupcakes with Hidden Cadbu...,https://www.pinkwhen.com/nutella-buttercream-c...,https://img.spoonacular.com/recipes/991625-556...,241.29,10.57,2.52,20.72,10,Pyszne


In [8]:
# db tests
def run_system_health_check():
    print("STARTING SYSTEM TESTS...\n")
    
    # --- TEST 1: File Existence ---
    db_file = 'nowaste.db'
    if os.path.exists(db_file):
        print(f"PASS: Database file '{db_file}' exists.")
    else:
        print(f"FAIL: Database file '{db_file}' not found.")
        return # Stop tests if file is missing

    # --- TEST 2: Database Connection ---
    try:
        # Try to execute a simple SQL query
        session.execute(text("SELECT 1"))
        print("PASS: Database connection established.")
    except Exception as e:
        print(f"FAIL: Database connection failed. Error: {e}")
        return

    # --- TEST 3: CRUD Logic (Create, Read, Update, Delete) ---
    print("\nRunning CRUD Test (on temporary data)...")
    try:
        # 3.1 CREATE
        test_user = UserProfile(
            name="TestUnit_Ghost", 
            weight=100.0, 
            goal="test", 
            dietary_restrictions="none"
        )
        session.add(test_user)
        session.commit()
        
        # 3.2 READ
        retrieved_user = session.query(UserProfile).filter_by(name="TestUnit_Ghost").first()
        assert retrieved_user is not None, "Failed to retrieve created user"
        assert retrieved_user.weight == 100.0, "Weight mismatch" # type: ignore
        
        # 3.3 UPDATE
        retrieved_user.goal = "updated_goal" # type: ignore
        session.commit()
        updated_user = session.query(UserProfile).filter_by(name="TestUnit_Ghost").first()
        assert updated_user.goal == "updated_goal", "Update failed" # type: ignore
        
        # 3.4 DELETE (Cleanup)
        session.delete(updated_user)
        session.commit()
        deleted_user = session.query(UserProfile).filter_by(name="TestUnit_Ghost").first()
        assert deleted_user is None, "Delete failed"
        
        print("PASS: CRUD operations working correctly.")
        
    except AssertionError as ae:
        print(f"FAIL: Assertion failed: {ae}")
        session.rollback()
    except Exception as e:
        print(f"FAIL: CRUD Error: {e}")
        session.rollback()

    # --- TEST 4: Verify Your Real Profile ---
    print("\nVerifying Active User Profile...")
    real_user = session.query(UserProfile).first()
    if real_user:
        print(f"PASS: Active user found: '{real_user.name}'")
        print(f"\tDetails: {real_user.weight}kg, Goal: {real_user.goal}")
    else:
        print("WARNING: No active user profile found. (Did you click 'Save Profile' in the previous step?)")

    print("\nTESTS COMPLETED.")

# Run the tests
run_system_health_check()

STARTING SYSTEM TESTS...

PASS: Database file 'nowaste.db' exists.
PASS: Database connection established.

Running CRUD Test (on temporary data)...
PASS: CRUD operations working correctly.

Verifying Active User Profile...

TESTS COMPLETED.


In [6]:
class IdentifiedItem(BaseModel):
    name_en_clean: str = Field(
        description="ONLY the core ingredient keyword in English "
                    "(e.g. 'eggs', 'milk', 'chicken breast'). "
                    "NO packaging, brands, or containers."
    )
    name_en: str = Field(
        description="Ingredient name in English, human-readable."
    )
    name_pl: str = Field(
        description="Ingredient name in Polish (for UI)."
    )
    quantity_estimated: Optional[str] = Field(
        description="Estimated quantity in Polish (e.g. '2 sztuki', 'ok. 200g'). "
                    "Use null if not possible."
    )
    confidence_level: Literal["high", "medium", "low"] = Field(
        description="Confidence level of visual recognition."
    )
    needs_clarification: bool = Field(
        description="True if the ingredient is ambiguous or unclear."
    )
    is_staple: bool = Field(
        description="True if ingredient is a basic kitchen staple "
                    "(spices, oils, sauces, sugar, flour, etc.)."
    )


class FridgeAnalysis(BaseModel):
    identified_items: List[IdentifiedItem]
    clarification_questions: List[str]
    ready_to_search: bool = Field(
        description="True ONLY if no clarification is needed and "
                    "core ingredients are confidently identified."
    )

STAPLES = {
    # spices
    "salt", "pepper", "black pepper", "white pepper",
    "paprika", "smoked paprika", "chili powder",
    "cumin", "curry powder", "turmeric",
    "oregano", "basil", "thyme", "rosemary",
    "bay leaf", "garlic powder", "onion powder",

    # fats
    "oil", "olive oil", "vegetable oil", "rapeseed oil",
    "sunflower oil", "coconut oil",
    "butter", "margarine", "ghee",

    # sweeteners
    "sugar", "brown sugar", "powdered sugar",
    "honey", "maple syrup", "agave syrup", "sweetener",

    # sauces
    "soy sauce", "ketchup", "mustard", "mayonnaise",
    "bbq sauce", "hot sauce", "sriracha",
    "vinegar", "apple cider vinegar", "balsamic vinegar",

    # technical
    "baking powder", "baking soda", "yeast",
    "cornstarch", "gelatin", "flour", "wheat flour"
}


def create_openai_file(file_path: str) -> str:
    """
    Uploads a local image file to OpenAI Files API for vision purposes.
    Returns file_id.
    """
    with open(file_path, "rb") as file_content:
        result = client.files.create(
            file=file_content,
            purpose="vision",
        )
    return result.id


def analyze_fridge_process(file_id: str) -> Optional[FridgeAnalysis]:
    """
    Initial fridge analysis using GPT vision.
    Returns FridgeAnalysis or None on error.
    """

    system_prompt = """
    Jesteś inteligentnym asystentem kulinarnym analizującym zdjęcie lodówki lub blatu.

    TWOJE ZADANIE:
    1. Zidentyfikuj wszystkie widoczne produkty spożywcze.
    2. Każdy produkt opisz jako osobny obiekt.

    ZASADY NAZW:
    - Pole `name_en_clean` MUSI zawierać WYŁĄCZNIE główny składnik kulinarny po angielsku
    (np. 'eggs', 'milk', 'chicken breast', 'bell pepper').
    - NIGDY nie dodawaj informacji o opakowaniu, marce ani formie
    ('carton', 'jar', 'package').
    - `name_en` może być bardziej opisowe, ale zgodne z tym samym składnikiem.
    - `name_pl` ma być przyjazne dla użytkownika.

    STAPLES (POMIJANE PRZY WYSZUKIWANIU PRZEPISÓW):
    Oznacz `is_staple = true` dla produktów, które:
    - są przyprawami (sól, pieprz, papryka, oregano, curry)
    - są olejami lub tłuszczami (olej, oliwa, masło)
    - są sosami (ketchup, musztarda, majonez, sos sojowy)
    - są słodzikami (cukier, miód, syrop klonowy)
    - są produktami technicznymi (mąka, drożdże, proszek do pieczenia)

    ILOŚĆ:
    - Podaj realistyczne oszacowanie ilości po polsku.
    - Jeśli to niemożliwe, użyj null.

    PEWNOŚĆ I PYTANIA:
    - Jeśli nie masz pewności, ustaw confidence_level na 'medium' lub 'low'.
    - Jeśli produkt jest niejednoznaczny (np. zawartość słoika),
    ustaw needs_clarification = true i dodaj pytanie po polsku.

    WARUNEK GOTOWOŚCI:
    - ready_to_search = true TYLKO jeśli:
    • brak otwartych pytań
    • kluczowe składniki mają confidence 'high' lub 'medium'
    """

    user_prompt = "Zidentyfikuj produkty spożywcze widoczne na zdjęciu."

    msg = [
        {"role": "developer", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": user_prompt},
                {"type": "input_image", "file_id": file_id},
            ],
        }
    ]

    try:
        response = client.responses.parse(
            model="gpt-5-mini",
            input=msg,
            text_format=FridgeAnalysis
        )

        return response.output_parsed

    except Exception as e:
        print(f"[ERROR] Fridge analysis failed: {e}")
        return None


def refine_analysis_process(
    file_id: str,
    previous_analysis: FridgeAnalysis,
    user_answers: str
) -> Optional[FridgeAnalysis]:
    """
    Refinement loop: updates analysis based on user answers.
    """

    system_prompt = """
    Jesteś inteligentnym asystentem kulinarnym.

    Użytkownik odpowiedział na Twoje pytania dotyczące poprzedniej analizy zdjęcia.

    ZASADY:
    - Zachowaj wszystkie poprawnie zidentyfikowane składniki.
    - Popraw lub uzupełnij TYLKO elementy, które były niejasne.
    - Nie dodawaj nowych produktów, jeśli nie wynikają z obrazu lub odpowiedzi użytkownika.

    NAZWY:
    - `name_en_clean` MUSI pozostać czystą nazwą składnika (bez opakowań).
    - Staples oznacz jako `is_staple = true`.

    CEL:
    - Zwróć kompletny obiekt typu FridgeAnalysis.
    - Jeśli wszystkie kluczowe składniki są już jasne,
    ustaw ready_to_search = true.
    """

    user_content = (
        f"Poprzednia analiza (JSON):\n{previous_analysis.model_dump_json()}\n\n"
        f"Odpowiedzi użytkownika:\n{user_answers}\n\n"
        "Zaktualizuj analizę."
    )

    msg = [
        {"role": "developer", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": user_content},
                {"type": "input_image", "file_id": file_id},
            ],
        }
    ]

    try:
        response = client.responses.parse(
            model="gpt-5-mini",
            input=msg,
            text_format=FridgeAnalysis
        )

        return response.output_parsed

    except Exception as e:
        print(f"[ERROR] Refinement failed: {e}")
        return None


In [ ]:
# --- Zmienne globalne do przechowywania stanu analizy ---
current_file_id = None
current_analysis_result = None

# --- Widgety UI ---
btn_upload = widgets.FileUpload(accept='image/*', multiple=False, description='Wgraj Zdjęcie')
output_vision = widgets.Output()

# Sekcja Chatu (domyślnie ukryta)
lbl_chat = widgets.HTML("<b>AI ma pytania:</b>")
txt_answer = widgets.Textarea(placeholder='Odpisz tutaj...', layout=widgets.Layout(width='100%', height='60px'))
btn_send_answer = widgets.Button(description='Wyślij odpowiedź', button_style='info', icon='paper-plane')
btn_skip = widgets.Button(description='Pomiń (Gotuj z tego co pewne)', button_style='warning')
box_chat = widgets.VBox([lbl_chat, txt_answer, widgets.HBox([btn_send_answer, btn_skip])])
box_chat.layout.display = 'none'

def display_fridge_result(result: FridgeAnalysis):
    """Renderuje wyniki analizy w ładny sposób."""
    if not result: return

    # Rozdzielamy składniki na główne i bazowe (staples)
    main_items = [i for i in result.identified_items if not i.is_staple and not i.needs_clarification]
    staple_items = [i for i in result.identified_items if i.is_staple]
    unclear_items = [i for i in result.identified_items if i.needs_clarification]

    html = "<h3>Wynik Analizy</h3>"
    
    # 1. Główne składniki
    if main_items:
        html += "<h4>Główne Produkty (Do bazy przepisu):</h4><ul>"
        for item in main_items:
            qty = f" ({item.quantity_estimated})" if item.quantity_estimated else ""
            html += f"<li><b>{item.name_pl}</b> {qty} <span style='color:gray; font-size:0.8em'>[{item.name_en_clean}]</span></li>"
        html += "</ul>"
    
    # 2. Staples (Przyprawy, oleje itp.)
    if staple_items:
        html += "<h4>Spiżarnia (Przyprawy/Dodatki):</h4><div style='color:#555; font-size:0.9em;'>"
        html += ", ".join([item.name_pl for item in staple_items])
        html += "</div>"

    # 3. Status
    if result.ready_to_search:
        html += "<div style='margin-top:10px; padding:10px; background:#e8f5e9; color:green; border-radius:5px;'><b>STATUS: GOTOWY DO SZUKANIA PRZEPISÓW</b></div>"
        box_chat.layout.display = 'none'
    else:
        html += "<div style='margin-top:10px; padding:10px; background:#fff3e0; color:#e65100; border-radius:5px;'><b>STATUS: WYMAGANE DOPYTANIE</b></div>"
        
        # Wyświetl pytania
        q_list = "".join([f"<li>{q}</li>" for q in result.clarification_questions])
        lbl_chat.value = f"<b>Pytania od AI:</b><ul>{q_list}</ul>"
        box_chat.layout.display = 'block'

    display(HTML(html))

def on_upload(change):
    """Obsługa wgrania pliku"""
    global current_file_id, current_analysis_result
    output_vision.clear_output()
    box_chat.layout.display = 'none'
    txt_answer.value = ""

    if not btn_upload.value: return

    with output_vision:
        # 1. Pobierz plik
        upl_file = btn_upload.value[0] if isinstance(btn_upload.value, tuple) else list(btn_upload.value.values())[0]
        content = upl_file['content']
        
        # 2. Pokaż zdjęcie
        display(Image(data=content, width=300))
        print("Analizuję zdjęcie (gpt-5-mini)...")
        
        # 3. Zapisz temp i wyślij
        temp_name = "temp_fridge.jpg"
        with open(temp_name, "wb") as f: f.write(content)
        
        try:
            current_file_id = create_openai_file(temp_name)
            current_analysis_result = analyze_fridge_process(current_file_id)
            
            clear_output(wait=True)
            display(Image(data=content, width=300))
            display_fridge_result(current_analysis_result)
            
        except Exception as e:
            print(f"Błąd: {e}")
        finally:
            if os.path.exists(temp_name): os.remove(temp_name)

def on_reply(b):
    """Obsługa odpowiedzi na pytania"""
    global current_analysis_result
    if not current_file_id or not txt_answer.value: return
    
    with output_vision:
        print("Aktualizuję analizę...")
        new_result = refine_analysis_process(current_file_id, current_analysis_result, txt_answer.value)
        if new_result:
            current_analysis_result = new_result
            clear_output(wait=True)
            # Ponowne wyświetlenie zdjęcia (żeby nie zniknęło)
            display_fridge_result(new_result)

def on_skip(b):
    """Wymuszenie gotowości (ignorowanie pytań)"""
    global current_analysis_result
    if current_analysis_result:
        current_analysis_result.ready_to_search = True
        with output_vision:
            clear_output(wait=True)
            display_fridge_result(current_analysis_result)

# Podpięcie zdarzeń
btn_upload.observe(on_upload, names='value')
btn_send_answer.on_click(on_reply)
btn_skip.on_click(on_skip)

display(widgets.VBox([
    widgets.HTML("<h3>Krok 2: Analiza Lodówki</h3>"),
    btn_upload,
    output_vision,
    box_chat
]))

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import json
import requests
from pydantic import BaseModel
from typing import List

# --- 0. KONFIGURACJA i CZYSZCZENIE ---
btn_generate = widgets.Button(
    description='🚀 Generuj Przepisy', 
    button_style='success', 
    layout=widgets.Layout(width='300px')
)
output_final = widgets.Output()       
interaction_output = widgets.Output() 

# --- 1. MODELE I LOGIKA AI (Backend) ---
class RecipeSearchPlan(BaseModel):
    ingredients: list[str]
    reason: str 

class SearchPlans(BaseModel):
    plans: list[RecipeSearchPlan]

def plan_recipe_search(ingredients: list[str], user_profile=None) -> SearchPlans:
    profile_text = f"Goal: {user_profile.goal}, Restrictions: {user_profile.dietary_restrictions}" if user_profile else ""
    
    system_prompt = f"""
    You are a culinary expert.
    From the provided ingredients, create 3 reasonable ingredient sets
    (each max 4 ingredients) that could realistically form a tasty dish.

    Rules:
    - Ignore spices, oils, sauces etc. (assume user has them).
    - Prefer protein + carb + vegetable combinations.
    - Do NOT try to use all ingredients at once.
    - For each set try to use diffrent ingredients if it is possible.
    - Try to create sets for realistic dishes.
    - Provide a short 'reason' in Polish describing the dish idea (e.g. 'Jajecznica z warzywami').

    CRITICAL:
    - Ingredient overlap between sets must be minimal.
    - Each ingredient can appear in MAX 1 set unless unavoidable.
    - Each set must represent a DIFFERENT type of dish:
    (e.g. breakfast, lunch, dinner, snack)
    {profile_text}
    """
    
    try:
        response = client.responses.parse(
            model="gpt-5-mini", # lub gpt-4o
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": json.dumps({"ingredients": ingredients})}
            ],
            text_format=SearchPlans,
            temperature=0.85
        )
        return response.output_parsed
    except Exception as e:
        print(f"Błąd AI: {e}")
        return None
    

# --- 2. LOGIKA SPOONACULAR (API) ---
def get_nutrient(rec, name) -> float:
    for n in rec.get("nutrition", {}).get("nutrients", []):
        if n.get("name") == name: return n.get("amount", 0)
    return 0

def fetch_spoonacular_recipes(plans: SearchPlans, user_profile):
    results_list = []
    
    find_params = {
        "apiKey": SPOONACULAR_API_KEY,
        "number": 5,
        "ranking": 2,
        "ignorePantry": True
    }

    api_limit_reached = False

    for i, plan in enumerate(plans.plans):
        if api_limit_reached: break
        
        safe_ings = plan.ingredients[:4]
        plan.ingredients = safe_ings
        
        current_find_params = find_params.copy()
        current_find_params['ingredients'] = ",".join(safe_ings)
        
        try:
            r_find = requests.get("https://api.spoonacular.com/recipes/findByIngredients", params=current_find_params)
            r_find.raise_for_status()
            find_data = r_find.json()
            if not find_data: continue

            recipe_ids = [str(item['id']) for item in find_data]
            ids_string = ",".join(recipe_ids)
            
            bulk_params = {
                "apiKey": SPOONACULAR_API_KEY,
                "ids": ids_string,
                "includeNutrition": True
            }
            
            r_bulk = requests.get("https://api.spoonacular.com/recipes/informationBulk", params=bulk_params)
            r_bulk.raise_for_status()
            bulk_data = r_bulk.json()
            
            valid_recipes = []
            for recipe in bulk_data:
                if not recipe.get('analyzedInstructions') and not recipe.get('instructions'):
                    continue

                cal = get_nutrient(recipe, "Calories")
                prot = get_nutrient(recipe, "Protein")
                
                if user_profile:
                    if user_profile.goal == 'muscle_gain' and prot < 20: continue
                    elif user_profile.goal == 'weight_loss' and cal > 800: continue

                valid_recipes.append(recipe)
            
            if valid_recipes:
                results_list.append((plan, valid_recipes[:2]))

        except requests.exceptions.HTTPError as e:
            if e.response.status_code in [402, 429]:
                print(f"⛔ LIMIT API PRZEKROCZONY (Kod {e.response.status_code})")
                api_limit_reached = True
            else:
                print(f"⚠️ Błąd HTTP: {e}")

        except Exception as e:
            print(f"Błąd API: {e}")
            
    return results_list

# --- 3. GENEROWANIE LISTY ZAKUPÓW (NOWOŚĆ) ---
def generate_shopping_list(recipe):
    """
    Porównuje składniki przepisu z zawartością lodówki.
    Zwraca dwie listy: (posiadane, brakujace).
    """
    have_items = []
    missing_items = []
    
    # 1. Pobieramy to co mamy w lodówce (zmienna globalna z Kroku 2)
    fridge_clean_names = set()
    if 'current_analysis_result' in globals() and current_analysis_result:
        for item in current_analysis_result.identified_items:
            fridge_clean_names.add(item.name_en_clean.lower())
            
    # Dodajemy podstawowe produkty (sól, woda, olej), zakładamy że user je ma
    pantry_staples = {'salt', 'pepper', 'water', 'oil', 'olive oil', 'sugar'}
    
    # 2. Iterujemy po składnikach przepisu
    # Spoonacular zwraca 'extendedIngredients'
    ingredients = recipe.get('extendedIngredients', [])
    
    for ing in ingredients:
        ing_name = (ing.get('nameClean') or ing.get('name', '')).lower()
        original_string = ing.get('original', ing_name)
        
        # Proste dopasowanie (czy nazwa z przepisu jest w lodówce lub odwrotnie)
        # np. "chicken breast" zawiera "chicken"
        is_present = False
        
        # Sprawdzamy staples
        if ing_name in pantry_staples:
            is_present = True
        else:
            # Sprawdzamy lodówkę
            for f_item in fridge_clean_names:
                if f_item in ing_name or ing_name in f_item:
                    is_present = True
                    break
        
        if is_present:
            have_items.append(original_string)
        else:
            missing_items.append(original_string)
            
    return have_items, missing_items

# --- 4. INTERFEJS: ETAP ZAPISU I OCENY ---
def save_meal_to_db(recipe, rating, notes):
    try:
        log = MealLog(
            recipe_name=recipe['title'],
            recipe_url=recipe['sourceUrl'],
            image_url=recipe['image'],
            calories=get_nutrient(recipe, "Calories"),
            protein=get_nutrient(recipe, "Protein"),
            carbs=get_nutrient(recipe, "Carbohydrates"),
            fat=get_nutrient(recipe, "Fat"),
            user_rating=rating,
            user_notes=notes
        )
        session.add(log)
        session.commit()
        return True
    except Exception as e:
        session.rollback()
        return False

def show_rating_stage(recipe):
    interaction_output.clear_output()
    with interaction_output:
        display(HTML(f"<h3 style='color:#27ae60;'>⭐ Oceń: {recipe['title']}</h3>"))
        w_rating = widgets.IntSlider(value=5, min=1, max=10, description='Smak:')
        w_notes = widgets.Textarea(placeholder='Notatki...', description='Notatki:')
        btn_save = widgets.Button(description="Zapisz", button_style='success', icon='save')
        
        def on_save(b):
            rating_val = w_rating.value
            notes_val = w_notes.value
            if save_meal_to_db(recipe, w_rating.value, w_notes.value):
                interaction_output.clear_output()
                with interaction_output:
                    display(HTML(f"""
                    <div style='background:#d4edda; padding:20px; border-radius:10px; border: 1px solid #c3e6cb; color:#155724;'>
                        <h3 style="margin-top:0;">✅ Zapisano pomyślnie!</h3>
                        <p>Danie <b>{recipe['title']}</b> zostało dodane do dziennika.</p>
                        <hr style="border-top: 1px solid #c3e6cb;">
                        <p><b>Twoja ocena:</b> {rating_val}/10 ⭐</p>
                        <p><b>Twój komentarz:</b> <i>"{notes_val if notes_val else '(brak)'}"</i></p>
                    </div>
                    """))
        
        btn_save.on_click(on_save)
        display(widgets.VBox([w_rating, w_notes, btn_save]))

# --- 5. INTERFEJS: ETAP GOTOWANIA + ZAKUPY (ZMODYFIKOWANY) ---
def show_cooking_stage(recipe):
    output_final.clear_output()
    interaction_output.clear_output()
    
    # Generujemy listę zakupów
    have, missing = generate_shopping_list(recipe)
    
    with interaction_output:
        # 1. Nagłówek i Link
        html_header = f"""
        <div style="border:2px solid #3498db; border-radius:10px; padding:20px; background:#f0f8ff; margin-bottom:20px;">
            <h2 style="margin-top:0; color:#2980b9;">👨‍🍳 Gotujesz: {recipe['title']}</h2>
            <div style="display:flex; gap:20px; align_items:center;">
                <img src="{recipe['image']}" style="width:150px; border-radius:10px;">
                <div>
                    <a href="{recipe['sourceUrl']}" target="_blank" style="background:#e74c3c; color:white; padding:12px 25px; text-decoration:none; border-radius:5px; font-weight:bold; font-size:16px;">
                        🔗 ZOBACZ INSTRUKCJĘ PRZYGOTOWANIA
                    </a>
                    <p style="margin-top:10px; color:#555;">Kliknij, aby otworzyć pełny przepis.</p>
                </div>
            </div>
        </div>
        """
        display(HTML(html_header))
        
        # 2. Lista Zakupów (Dwie kolumny)
        # Tworzymy HTML dla list
        list_have_html = "".join([f"<li style='color:#27ae60;'>✅ {item}</li>" for item in have])
        list_missing_html = "".join([f"<li style='color:#c0392b;'>🛒 <b>{item}</b></li>" for item in missing])
        
        html_lists = f"""
        <div style="display:flex; gap:20px; margin-bottom:20px;">
            <div style="flex:1; padding:15px; border:1px solid #ddd; border-radius:8px; background:white;">
                <h4 style="margin-top:0; color:#27ae60;">👍 Masz w domu (lub baza):</h4>
                <ul style="padding-left:20px;">{list_have_html}</ul>
            </div>
            <div style="flex:1; padding:15px; border:1px solid #e74c3c; border-radius:8px; background:#fff5f5;">
                <h4 style="margin-top:0; color:#c0392b;">📝 Lista Zakupów (Brakuje):</h4>
                <ul style="padding-left:20px;">{list_missing_html}</ul>
            </div>
        </div>
        """
        display(HTML(html_lists))
        
        # 3. Przycisk Zakończenia
        btn_eaten = widgets.Button(
            description="Ugotowane i zjedzone! -> Oceń", 
            button_style='primary', 
            layout=widgets.Layout(width='100%', height='50px')
        )
        btn_eaten.on_click(lambda b: show_rating_stage(recipe))
        display(btn_eaten)

# --- 6. INTERFEJS: KARTA PRZEPISU ---
def create_card(recipe):
    cal = int(get_nutrient(recipe, "Calories"))
    prot = int(get_nutrient(recipe, "Protein"))
    title = recipe['title']
    
    html = widgets.HTML(f"""
    <div style="width:280px; border:1px solid #ddd; border-radius:10px; overflow:hidden; background:white; margin-bottom:10px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
        <div style="height:150px; overflow:hidden;">
            <img src="{recipe['image']}" style="width:100%; height:100%; object-fit:cover;">
        </div>
        <div style="padding:12px;">
            <h4 title="{title}" style="margin: 0 0 8px 0; font-size: 16px; line-height: 1.3em; min-height: 42px; display: -webkit-box; -webkit-line-clamp: 2; -webkit-box-orient: vertical; overflow: hidden;">
                {title}
            </h4>
            <div style="color:#555; font-size:13px; display:flex; justify-content:space-between;">
                <span>🔥 {cal} kcal</span>
                <span>🥩 {prot}g białka</span>
            </div>
        </div>
    </div>
    """)
    
    btn = widgets.Button(
        description="Gotuję to!", 
        button_style='info', 
        layout=widgets.Layout(width='95%', margin='0 auto 10px auto')
    )
    btn.on_click(lambda b: show_cooking_stage(recipe))
    
    return widgets.VBox([html, btn], layout=widgets.Layout(margin='0 15px 20px 0'))

# --- 7. GŁÓWNA PĘTLA ---
def on_generate_click(b):
    output_final.clear_output()
    interaction_output.clear_output()
    
    if 'current_analysis_result' not in globals() or not current_analysis_result:
        with output_final: print("⚠️ Brak analizy (Krok 2).")
        return

    ingredients = [i.name_en_clean for i in current_analysis_result.identified_items 
                   if not i.is_staple and not i.needs_clarification]
    
    user = session.query(UserProfile).first() if 'session' in globals() else None

    with output_final:
        print("🧠 AI planuje...")
        plans = plan_recipe_search(ingredients, user)
        if not plans: return
        
        # Debug
        d_html = "<div style='background:#f8f9fa; padding:10px; border-left:4px solid #f1c40f; margin-bottom:15px; font-size:12px;'>"
        for p in plans.plans: d_html += f"<b>Plan:</b> {', '.join(p.ingredients)} <i style='color:#666'>({p.reason})</i><br>"
        d_html += "</div>"
        display(HTML(d_html))

        print("🌍 Szukam (findByIngredients + bulk)...")
        results = fetch_spoonacular_recipes(plans, user)
        
        if not results:
            print("❌ Brak wyników.")
            return

        layout_children = []
        for plan, recipes in results:
            layout_children.append(widgets.HTML(f"<h3 style='border-bottom:1px solid #eee; margin-top:25px; color:#2c3e50;'>💡 {plan.reason}</h3>"))
            cards = [create_card(r) for r in recipes]
            layout_children.append(widgets.HBox(cards, layout=widgets.Layout(flex_flow='row wrap')))
            
        display(widgets.VBox(layout_children))

btn_generate.on_click(on_generate_click)

display(widgets.VBox([
    widgets.HTML("<h3>Krok 3: Wybór Przepisu i Lista Zakupów</h3>"),
    btn_generate,
    output_final,
    widgets.HTML("<hr>"),
    interaction_output
]))

In [ ]:
# import ipywidgets as widgets
# from IPython.display import display, HTML, clear_output
# import json
# import requests
# from pydantic import BaseModel
# from typing import List

# # --- 0. KONFIGURACJA i CZYSZCZENIE ---
# btn_generate = widgets.Button(
#     description='🚀 Generuj Przepisy', 
#     button_style='success', 
#     layout=widgets.Layout(width='300px')
# )
# output_final = widgets.Output()       
# interaction_output = widgets.Output() 

# # --- 1. MODELE I LOGIKA AI (Backend) ---
# class RecipeSearchPlan(BaseModel):
#     ingredients: list[str]
#     reason: str 

# class SearchPlans(BaseModel):
#     plans: list[RecipeSearchPlan]

# def plan_recipe_search(ingredients: list[str], user_profile=None) -> SearchPlans:
#     profile_text = f"Goal: {user_profile.goal}, Restrictions: {user_profile.dietary_restrictions}" if user_profile else ""
    
#     system_prompt = f"""
#     You are a culinary expert.
#     From the provided ingredients, create 3 reasonable ingredient sets
#     (each max 4 ingredients) that could realistically form a tasty dish.

#     Rules:
#     - Ignore spices, oils, sauces etc. (assume user has them).
#     - Prefer protein + carb + vegetable combinations.
#     - Do NOT try to use all ingredients at once.
#     - For each set try to use diffrent ingredients if it is possible.
#     - Try to create sets for realistic dishes.
#     - Provide a short 'reason' in Polish describing the dish idea (e.g. 'Jajecznica z warzywami').
#     {profile_text}
#     """
    
#     try:
#         response = client.responses.parse(
#             model="gpt-5-mini", # lub gpt-4o
#             input=[
#                 {"role": "system", "content": system_prompt},
#                 {"role": "user", "content": json.dumps({"ingredients": ingredients})}
#             ],
#             text_format=SearchPlans
#         )
#         return response.output_parsed
#     except Exception as e:
#         print(f"Błąd AI: {e}")
#         return None
    

# # --- 2. LOGIKA SPOONACULAR (API) ---
# def get_nutrient(rec, name) -> float:
#     for n in rec.get("nutrition", {}).get("nutrients", []):
#         if n.get("name") == name: return n.get("amount", 0)
#     return 0

# def fetch_spoonacular_recipes(plans: SearchPlans, user_profile):
#     results_list = []
    
#     # KROK 1: Parametry wyszukiwania
#     find_params = {
#         "apiKey": SPOONACULAR_API_KEY,
#         "number": 5,
#         "ranking": 2,
#         "ignorePantry": True
#     }

#     api_limit_reached = False

#     for i, plan in enumerate(plans.plans):
#         if api_limit_reached: break
        
#         safe_ings = plan.ingredients[:4]
#         plan.ingredients = safe_ings
        
#         current_find_params = find_params.copy()
#         current_find_params['ingredients'] = ",".join(safe_ings)
        
#         try:
#             # 1. Szukamy pomysłów
#             r_find = requests.get("https://api.spoonacular.com/recipes/findByIngredients", params=current_find_params)
#             r_find.raise_for_status()
#             find_data = r_find.json()
#             if not find_data: continue

#             # 2. Pobieramy szczegóły (Bulk)
#             recipe_ids = [str(item['id']) for item in find_data]
#             ids_string = ",".join(recipe_ids)
            
#             bulk_params = {
#                 "apiKey": SPOONACULAR_API_KEY,
#                 "ids": ids_string,
#                 "includeNutrition": True
#             }
            
#             r_bulk = requests.get("https://api.spoonacular.com/recipes/informationBulk", params=bulk_params)
#             r_bulk.raise_for_status()
#             bulk_data = r_bulk.json()
            
#             # 3. Filtrowanie
#             valid_recipes = []
#             for recipe in bulk_data:
#                 if not recipe.get('analyzedInstructions') and not recipe.get('instructions'):
#                     continue

#                 cal = get_nutrient(recipe, "Calories")
#                 prot = get_nutrient(recipe, "Protein")
                
#                 if user_profile:
#                     if user_profile.goal == 'muscle_gain' and prot < 20: continue
#                     elif user_profile.goal == 'weight_loss' and cal > 1200: continue

#                 valid_recipes.append(recipe)
            
#             if valid_recipes:
#                 results_list.append((plan, valid_recipes[:2])) # Max 2 wyniki na plan

#         except requests.exceptions.HTTPError as e:
#             if e.response.status_code in [402, 429]:
#                 print(f"⛔ LIMIT API PRZEKROCZONY (Kod {e.response.status_code}) przy: {safe_ings}")
#                 api_limit_reached = True
#             else:
#                 print(f"⚠️ Błąd HTTP: {e}")

#         except Exception as e:
#             print(f"Błąd API: {e}")
            
#     return results_list

# # --- 3. INTERFEJS: ETAP ZAPISU I OCENY ---
# def save_meal_to_db(recipe, rating, notes):
#     try:
#         log = MealLog(
#             recipe_name=recipe['title'],
#             recipe_url=recipe['sourceUrl'],
#             image_url=recipe['image'],
#             calories=get_nutrient(recipe, "Calories"),
#             protein=get_nutrient(recipe, "Protein"),
#             carbs=get_nutrient(recipe, "Carbohydrates"),
#             fat=get_nutrient(recipe, "Fat"),
#             user_rating=rating,
#             user_notes=notes
#         )
#         session.add(log)
#         session.commit()
#         return True
#     except Exception as e:
#         session.rollback()
#         return False

# def show_rating_stage(recipe):
#     interaction_output.clear_output()
#     with interaction_output:
#         display(HTML(f"<h3 style='color:#27ae60;'>⭐ Oceń: {recipe['title']}</h3>"))
#         w_rating = widgets.IntSlider(value=5, min=1, max=10, description='Smak:')
#         w_notes = widgets.Textarea(placeholder='Notatki...', description='Notatki:')
#         btn_save = widgets.Button(description="Zapisz", button_style='success', icon='save')
        
#         def on_save(b):
#             rating_val = w_rating.value
#             notes_val = w_notes.value
#             if save_meal_to_db(recipe, w_rating.value, w_notes.value):
#                 interaction_output.clear_output()
#                 with interaction_output:
#                     display(HTML(f"""
#                     <div style='background:#d4edda; padding:20px; border-radius:10px; border: 1px solid #c3e6cb; color:#155724;'>
#                         <h3 style="margin-top:0;">✅ Zapisano pomyślnie!</h3>
#                         <p>Danie <b>{recipe['title']}</b> zostało dodane do dziennika.</p>
#                         <hr style="border-top: 1px solid #c3e6cb;">
#                         <p><b>Twoja ocena:</b> {rating_val}/10 ⭐</p>
#                         <p><b>Twój komentarz:</b> <i>"{notes_val if notes_val else '(brak)'}"</i></p>
#                     </div>
#                     """))
        
#         btn_save.on_click(on_save)
#         display(widgets.VBox([w_rating, w_notes, btn_save]))

# # --- 4. INTERFEJS: ETAP GOTOWANIA ---
# def show_cooking_stage(recipe):
#     output_final.clear_output()
#     interaction_output.clear_output()
#     with interaction_output:
#         html = f"""
#         <div style="border:2px solid #3498db; border-radius:10px; padding:20px; background:#f0f8ff;">
#             <h2 style="margin-top:0; color:#2980b9;">👨‍🍳 Gotujesz: {recipe['title']}</h2>
#             <div style="display:flex; gap:20px;">
#                 <img src="{recipe['image']}" style="width:200px; border-radius:10px;">
#                 <div>
#                     <a href="{recipe['sourceUrl']}" target="_blank" style="background:#e74c3c; color:white; padding:10px 20px; text-decoration:none; border-radius:5px; font-weight:bold; display:inline-block; margin-bottom:10px;">
#                         🔗 OTWÓRZ PRZEPIS
#                     </a>
#                 </div>
#             </div>
#         </div>
#         """
#         display(HTML(html))
#         btn_eaten = widgets.Button(description="Już zjadłem/am - Oceń", button_style='primary', layout=widgets.Layout(width='100%'))
#         btn_eaten.on_click(lambda b: show_rating_stage(recipe))
#         display(btn_eaten)

# # --- 5. INTERFEJS: KARTA PRZEPISU (POPRAWIONA) ---
# def create_card(recipe):
#     """
#     Karta z elastyczną wysokością tytułu i tooltipem.
#     """
#     cal = int(get_nutrient(recipe, "Calories"))
#     prot = int(get_nutrient(recipe, "Protein"))
#     title = recipe['title'] # Pełny tytuł
    
#     # CSS: 
#     # - min-height: 50px (daje miejsce na 2-3 linie tekstu)
#     # - -webkit-line-clamp: 2 (ucina po 2 liniach i dodaje "...")
#     # - title="{title}" (systemowy tooltip po najechaniu myszką)
    
#     html = widgets.HTML(f"""
#     <div style="width:280px; border:1px solid #ddd; border-radius:10px; overflow:hidden; background:white; margin-bottom:10px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
        
#         <div style="height:150px; overflow:hidden;">
#             <img src="{recipe['image']}" style="width:100%; height:100%; object-fit:cover;">
#         </div>
        
#         <div style="padding:12px;">
#             <h4 title="{title}" style="
#                 margin: 0 0 8px 0; 
#                 font-size: 16px; 
#                 line-height: 1.3em;
#                 min-height: 42px; /* Miejsce na 2 linie */
#                 display: -webkit-box;
#                 -webkit-line-clamp: 2;
#                 -webkit-box-orient: vertical;
#                 overflow: hidden;
#             ">
#                 {title}
#             </h4>
            
#             <div style="color:#555; font-size:13px; display:flex; justify-content:space-between;">
#                 <span>🔥 {cal} kcal</span>
#                 <span>🥩 {prot}g białka</span>
#             </div>
#         </div>
#     </div>
#     """)
    
#     btn = widgets.Button(
#         description="Gotuję to!", 
#         button_style='info', 
#         layout=widgets.Layout(width='95%', margin='0 auto 10px auto') # Wyśrodkowany przycisk
#     )
#     btn.on_click(lambda b: show_cooking_stage(recipe))
    
#     return widgets.VBox([html, btn], layout=widgets.Layout(margin='0 15px 20px 0'))

# # --- 6. GŁÓWNA PĘTLA ---
# def on_generate_click(b):
#     output_final.clear_output()
#     interaction_output.clear_output()
    
#     if 'current_analysis_result' not in globals() or not current_analysis_result:
#         with output_final: print("⚠️ Brak analizy (Krok 2).")
#         return

#     ingredients = [i.name_en_clean for i in current_analysis_result.identified_items 
#                    if not i.is_staple and not i.needs_clarification]
    
#     user = session.query(UserProfile).first() if 'session' in globals() else None

#     with output_final:
#         print("🧠 AI planuje...")
#         plans = plan_recipe_search(ingredients, user)
#         if not plans: return
        
#         # Debug
#         d_html = "<div style='background:#f8f9fa; padding:10px; border-left:4px solid #f1c40f; margin-bottom:15px; font-size:12px;'>"
#         for p in plans.plans: d_html += f"<b>Plan:</b> {', '.join(p.ingredients)} <i style='color:#666'>({p.reason})</i><br>"
#         d_html += "</div>"
#         display(HTML(d_html))

#         print("🌍 Szukam (findByIngredients + bulk)...")
#         results = fetch_spoonacular_recipes(plans, user)
        
#         # clear_output(wait=True)
#         if not results:
#             print("❌ Brak wyników.")
#             return

#         layout_children = []
#         for plan, recipes in results:
#             layout_children.append(widgets.HTML(f"<h3 style='border-bottom:1px solid #eee; margin-top:25px; color:#2c3e50;'>💡 {plan.reason}</h3>"))
#             cards = [create_card(r) for r in recipes]
#             layout_children.append(widgets.HBox(cards, layout=widgets.Layout(flex_flow='row wrap')))
            
#         display(widgets.VBox(layout_children))

# btn_generate.on_click(on_generate_click)

# display(widgets.VBox([
#     widgets.HTML("<h3>Krok 3: Wybór Przepisu (Dwuetapowe API)</h3>"),
#     btn_generate,
#     output_final,
#     widgets.HTML("<hr>"),
#     interaction_output
# ]))

In [10]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import io
import base64


# --- 2. FUNKCJE POMOCNICZE UI ---
def get_progress_bar_html(current, target, label, unit="g", color="#3498db"):
    if target <= 0: target = 1 
    percent = min(100, int((current / target) * 100))
    return f"""
    <div style="margin-bottom:10px;">
        <div style="display:flex; justify-content:space-between; margin-bottom:2px; font-size:0.85em; color:#555;">
            <span>{label}</span>
            <span><b>{int(current)}</b> / {int(target)}{unit} ({percent}%)</span>
        </div>
        <div style="background:#ecf0f1; border-radius:8px; height:6px; width:100%;">
            <div style="background:{color}; width:{percent}%; height:100%; border-radius:8px;"></div>
        </div>
    </div>
    """

def create_chart_image(daily_data, target):
    """Wykres Tygodniowy"""
    dates = list(daily_data.keys())
    values = list(daily_data.values())
    labels = [datetime.strptime(d, "%Y-%m-%d").strftime("%d.%m") for d in dates]
    colors = ['#e74c3c' if v > target * 1.1 else '#3498db' for v in values]
    
    plt.figure(figsize=(6, 3))
    bars = plt.bar(labels, values, color=colors, alpha=0.8, width=0.6)
    
    plt.axhline(y=target, color='gray', linestyle='--', alpha=0.5, label=f'Cel: {target}')
    plt.title(f"Kalorie (Cel: {target} kcal)", fontsize=10, pad=10)
    plt.grid(axis='y', linestyle=':', alpha=0.3)
    plt.box(False)
    plt.tick_params(axis='both', which='both', length=0)
    
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            plt.text(bar.get_x() + bar.get_width()/2., height + 50,
                     f'{int(height)}', ha='center', va='bottom', fontsize=8, color='#333')

    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', dpi=100)
    plt.close()
    
    buf.seek(0)
    img_base64 = base64.b64encode(buf.read()).decode('utf-8')
    return f'<img src="data:image/png;base64,{img_base64}" style="width:100%; border-radius:5px;">'


# --- 3. GŁÓWNA FUNKCJA DASHBOARDU ---
btn_refresh = widgets.Button(description="🔄 Odśwież Dane", button_style='warning', icon='refresh')
output_stats = widgets.Output()

def render_dashboard(b=None):
    output_stats.clear_output()
    try: session.commit()
    except: session.rollback()

    try:
        df = pd.read_sql(session.query(MealLog).statement, session.bind)
    except Exception as e:
        with output_stats: print(f"❌ Błąd bazy: {e}")
        return

    user = session.query(UserProfile).first()
    
    # Pobieramy GOTOWE wartości z bazy (lub domyślne jeśli brak usera)
    if user:
        t_cal = user.target_calories
        t_prot = user.target_protein
        t_fat = user.target_fat
        t_carb = user.target_carbs
        
        # Odtwarzamy procenty tylko do wyświetlania na pasku (opcjonalne)
        # Możemy to obliczyć w locie bo to tylko display
        pct_prot = int((t_prot * 4 / t_cal) * 100) if t_cal else 0
        pct_fat = int((t_fat * 9 / t_cal) * 100) if t_cal else 0
        pct_carb = int((t_carb * 4 / t_cal) * 100) if t_cal else 0
    else:
        # Fallback
        t_cal, t_prot, t_fat, t_carb = 2000, 150, 70, 250
        pct_prot, pct_fat, pct_carb = 30, 30, 40
    
    with output_stats:
        if df.empty:
            display(HTML("<div style='padding:15px; background:#f9e79f;'>📭 Dziennik jest pusty. Zapisz posiłek w Kroku 3!</div>"))
            return
        
        try:
            df['dt'] = pd.to_datetime(df['date'])
            df['day_str'] = df['dt'].dt.strftime('%Y-%m-%d')
        except Exception as e:
            print(f"Błąd dat: {e}")
            return
            
        now = datetime.now()
        today_str = now.strftime('%Y-%m-%d')
        start_of_week = now - timedelta(days=6)
        start_of_week_str = start_of_week.strftime('%Y-%m-%d')

        df_today = df[df['day_str'] == today_str]
        df_week = df[df['day_str'] >= start_of_week_str].copy()
        
        # --- ZAKŁADKA 1: DZISIAJ ---
        def render_today():
            sum_cal = df_today['calories'].sum()
            sum_prot = df_today['protein'].sum()
            sum_fat = df_today['fat'].sum()
            sum_carb = df_today['carbs'].sum()
            
            html = f"<div style='padding:10px;'>"
            html += f"<h3 style='margin-top:0; color:#2c3e50;'>📅 Dzisiaj ({today_str})</h3>"
            
            # KAFELKI
            html += f"""
            <div style="display:flex; gap:10px; margin-bottom:20px; flex-wrap:wrap;">
                <div style="background:#e8f6f3; padding:10px; border-radius:8px; flex:1; min-width:80px; text-align:center; border:1px solid #d1f2eb;">
                    <div style="font-size:18px; font-weight:bold; color:#16a085;">{int(sum_cal)}</div>
                    <div style="font-size:11px; color:#666;">Kcal</div>
                </div>
                 <div style="background:#fef9e7; padding:10px; border-radius:8px; flex:1; min-width:80px; text-align:center; border:1px solid #f9e79f;">
                    <div style="font-size:18px; font-weight:bold; color:#f39c12;">{int(sum_prot)}g</div>
                    <div style="font-size:11px; color:#666;">Białko</div>
                </div>
                 <div style="background:#fff8e1; padding:10px; border-radius:8px; flex:1; min-width:80px; text-align:center; border:1px solid #ffe082;">
                    <div style="font-size:18px; font-weight:bold; color:#f57f17;">{int(sum_fat)}g</div>
                    <div style="font-size:11px; color:#666;">Tłuszcze</div>
                </div>
                 <div style="background:#f3e5f5; padding:10px; border-radius:8px; flex:1; min-width:80px; text-align:center; border:1px solid #e1bee7;">
                    <div style="font-size:18px; font-weight:bold; color:#8e24aa;">{int(sum_carb)}g</div>
                    <div style="font-size:11px; color:#666;">Węgle</div>
                </div>
            </div>
            """
            
            # PASKI POSTĘPU - TERAZ WSZYSTKIE MAJĄ OPIS CELU (%)
            html += get_progress_bar_html(sum_cal, t_cal, "🔥 Kalorie", unit="", color="#2ecc71")
            html += get_progress_bar_html(sum_prot, t_prot, f"🥩 Białko (Cel: {pct_prot}%)", color="#3498db")
            html += get_progress_bar_html(sum_fat, t_fat, f"🥑 Tłuszcze (Cel: {pct_fat}%)", color="#f1c40f")
            html += get_progress_bar_html(sum_carb, t_carb, f"🍞 Węglowodany (Cel: {pct_carb}%)", color="#9b59b6")
            
            return widgets.HTML(html + "</div>")

        # --- ZAKŁADKA 2: TYDZIEŃ ---
        def render_week():
            daily_groups_raw = df_week.groupby('day_str')['calories'].sum().to_dict()
            daily_data_full = {}
            for i in range(6, -1, -1):
                d = now - timedelta(days=i)
                key = d.strftime('%Y-%m-%d')
                daily_data_full[key] = daily_groups_raw.get(key, 0)

            total_week = sum(daily_data_full.values())
            avg_week = total_week / 7
            
            html = f"<div style='padding:10px;'>"
            html += f"<h3 style='margin-top:0; color:#2c3e50;'>📅 Ostatnie 7 Dni</h3>"
            html += f"<p>Łącznie: <b>{int(total_week)} kcal</b> | Średnia: <b>{int(avg_week)}</b> / dzień</p>"
            img_tag = create_chart_image(daily_data_full, t_cal)
            return widgets.HTML(html + "<div style='margin-top:20px; text-align:center;'>" + img_tag + "</div></div>")

        # --- ZAKŁADKA 3: HISTORIA ---
        def render_log_table():
            out = widgets.Output()
            with out:
                view_df = df.copy()
                view_df['Data'] = view_df['dt'].dt.strftime('%Y-%m-%d %H:%M')
                view_df = view_df[['Data', 'recipe_name', 'calories', 'protein', 'fat', 'carbs', 'user_rating']]
                view_df.columns = ['Data', 'Danie', 'Kcal', 'Białko', 'Tłuszcz', 'Węgle', 'Ocena']
                view_df = view_df.sort_values(by='Data', ascending=False).head(15)
                display(view_df.style.hide(axis="index").background_gradient(cmap='Blues', subset=['Kcal']))
            return out

        tabs = widgets.Tab(children=[render_today(), render_week(), render_log_table()])
        tabs.set_title(0, '📆 Dzisiaj')
        tabs.set_title(1, '📊 Tydzień')
        tabs.set_title(2, '📜 Historia')
        display(tabs)

btn_refresh.on_click(render_dashboard)
render_dashboard()

display(widgets.VBox([
    widgets.HTML("<h3>📊 Krok 4: Twój Dziennik i Statystyki</h3>"),
    btn_refresh,
    output_stats
]))